<h1 align="center">Sql Business Data Analysis_Services Company</h1>

In [3]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-0C9VGOP;"
    "DATABASE=service;"
    "Trusted_Connection=yes;"
)

print("Done")

Done


In [5]:
import warnings
warnings.filterwarnings('ignore') 
import pandas as pd
query = """
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
"""

tables = pd.read_sql(query, conn)

tables

,TABLE_NAME
0,service
1,sysdiagrams
2,clients
3,branches
4,employees


# Q1:Data Exploration
# How many services,clients,branches,and employees are in the database?


In [15]:
query = """
SELECT
    (SELECT COUNT(*) FROM dbo.service) AS Total_Services,
    (SELECT COUNT(*) FROM dbo.clients) AS Total_Clients,
    (SELECT COUNT(*) FROM dbo.branches) AS Total_Branches,
    (SELECT COUNT(*) FROM dbo.employees) AS Total_Employees;
"""

df = pd.read_sql(query, conn)
df

,Total_Services,Total_Clients,Total_Branches,Total_Employees
0,651,80,10,35


# Q2:Service Types
# What are the different types of services provideed by the company?

In [24]:
query = "SELECT DISTINCT Service_Type FROM dbo.service;"
df = pd.read_sql(query, conn)
df

,Service_Type
0,Consulting
1,Data Analysis
2,Financial Advisory
3,IT Support
4,IT Support
5,Marketing
6,Project Management
7,Training


# Q3:Total Revenue
# What is the total revenue generated from all services?

In [25]:
query = "SELECT ROUND(SUM(Total_Revenue), 2) AS Total_revenue FROM dbo.service;"
df = pd.read_sql(query, conn)
df

,Total_revenue
0,883476.48


# Q4:High-value services
# Which services generated more than $1000 in revenue?

In [28]:
query = "SELECT DISTINCT Service_Type FROM dbo.service WHERE Total_Revenue > 1000;"
df = pd.read_sql(query, conn)
df

,Service_Type
0,Consulting
1,Data Analysis
2,Financial Advisory
3,IT Support
4,IT Support
5,Marketing
6,Project Management
7,Training


# Q5:Service type performance
# which service type generates the highest total revenue?

In [30]:
query = """
SELECT Service_Type, ROUND(SUM(Total_Revenue), 2) AS Total_revenue 
FROM dbo.service 
GROUP BY Service_Type 
ORDER BY SUM(Total_Revenue) DESC;
"""
df = pd.read_sql(query, conn)
df

,Service_Type,Total_revenue
0,Financial Advisory,163341.94
1,Data Analysis,151048.47
2,Project Management,148704.97
3,Consulting,144057.24
4,IT Support,100561.71
5,Marketing,96329.62
6,Training,78113.85
7,IT Support,1318.68


# Q6:Branch Performance
# Which branch generates the highest total revenue?

In [31]:
query = """
SELECT TOP 1 b.branch_Name, ROUND(SUM(s.total_revenue), 2) AS Total_revenue 
FROM dbo.branches b 
JOIN dbo.service s ON b.branch_id = s.branch_id 
GROUP BY b.branch_name 
ORDER BY Total_revenue DESC;
"""
df = pd.read_sql(query, conn)
df

,branch_Name,Total_revenue
0,Dubai,104290.02


# Q7:Regional Performance
# Which region generates the highest total revenue?

In [32]:
query = """
SELECT TOP 1 b.Region, ROUND(SUM(s.total_revenue), 2) AS Total_revenue
FROM dbo.branches b
JOIN dbo.service s ON b.branch_id = s.branch_id
GROUP BY b.Region
ORDER BY Total_revenue DESC;
"""
df = pd.read_sql(query, conn)
df

,Region,Total_revenue
0,Europe,330810.85


# Q8:Branches and Number of Services
# Show every branch and the number of services provided by each branch, including branches that have no services.

In [38]:
query = """
SELECT 
    B.Branch_Name,
    COUNT(S.Service_ID) AS Number_of_Services
FROM dbo.branches B
LEFT JOIN dbo.service S 
    ON B.Branch_ID = S.Branch_ID
GROUP BY B.Branch_Name;
"""
df = pd.read_sql(query, conn)
df

,Branch_Name,Number_of_Services
0,Abu Dhabii,63
1,Alexandria,59
2,Birmingham,66
3,Cairo,67
4,Dubai,81
5,Giza,54
6,Leeds,60
7,London Central,65
8,Manchester,65
9,Sharjah,71


# Q9: High performance service types
# Which service types generated a total revenue greater than $100,000?

In [34]:
query = """
SELECT Service_Type, ROUND(SUM(Total_Revenue), 2) AS Total_Revenue
FROM dbo.service
GROUP BY Service_Type
HAVING SUM(Total_Revenue) > 100000;
"""
df = pd.read_sql(query, conn)
df

,Service_Type,Total_Revenue
0,Consulting,144057.24
1,Data Analysis,151048.47
2,Financial Advisory,163341.94
3,IT Support,100561.71
4,Project Management,148704.97


# Q10:Unused services
# Which services have never been used by any client?

In [35]:
query = """
SELECT s.service_type, c.client_id
FROM dbo.service s
LEFT JOIN dbo.clients c ON s.Client_ID = c.Client_ID
WHERE c.Client_ID IS NULL;
"""
df = pd.read_sql(query, conn)
df

,service_type,client_id
0,Consulting,None


# Q11:Customer segmentation
# How can clients be classified into Low, Medium, and High spending categories based on their total revenue?

In [36]:
query = """
SELECT c.Client_Name,
       ROUND(SUM(s.total_revenue), 2) AS total_revenue,
       CASE 
            WHEN SUM(s.total_revenue) > 50000 THEN 'high'
            WHEN SUM(s.total_revenue) BETWEEN 20000 AND 50000 THEN 'medium'
            ELSE 'low'
       END AS spending_category
FROM dbo.clients c
JOIN dbo.service s ON c.client_id = s.client_id
GROUP BY c.client_name;
"""
df = pd.read_sql(query, conn)
df

,Client_Name,total_revenue,spending_category
0,Amelia Taylor,8262.14,low
1,Amelia Wilson,14194.50,low
2,Arthur Ali,6759.84,low
3,Arthur Jones,9246.65,low
4,Arthur Walker,6941.74,low
...,...,...,...
69,William Lewis,12697.29,low
70,William Williams,11441.64,low
71,William Wilson,10860.27,low
72,Youssef Davies,8062.08,low


# Q12:High value clients by region
# Which regions have a total client revenue greater than $200,000?

In [37]:
query = """
SELECT b.region,
       ROUND(SUM(s.total_revenue), 2) AS Total_revenue
FROM dbo.branches b
JOIN dbo.service s ON b.branch_id = s.branch_id
GROUP BY b.region
HAVING SUM(s.total_revenue) > 200000;
"""
df = pd.read_sql(query, conn)
df

,region,Total_revenue
0,Europe,330810.85
1,Gulf,290170.74
2,Middle East,262494.89
